# Anadolu'da Üç Çağın Yerleşim Seçimi: Analiz Defteri

Bu defter bütün analizi Google Colab'da çalıştırır. Bilgisayarına hiçbir şey kurman gerekmez.

**Nasıl kullanılır?** Hücreleri **yukarıdan aşağıya, sırayla** çalıştır (her hücrenin solundaki ▶ düğmesi).
Bir hücre hata verirse sonrakine geçme; hata mesajını öğretmenine göster.

Rehber: *Adım Adım Rehber*, Aşama 4.

## 1. Kurulum
GitHub'daki son kodu indirir ve gerekli paketleri kurar (1–2 dakika).

In [ ]:
!rm -rf /content/tubitak-arkeoloji
!git clone -q https://github.com/kingmario1569-hub/tubitak-arkeoloji.git /content/tubitak-arkeoloji
%cd /content/tubitak-arkeoloji
!pip install -q -r requirements-analiz.txt
!git log -1 --format="Son commit: %h  %ad  %s" --date=short

## 2. Google Drive'daki coğrafi katmanları bağla
Drive'ında **`tubitak-veri`** adında bir klasör olmalı (Rehber, Aşama 3). İçinde şunlar bulunur:
```
tubitak-veri/
├── hydrorivers/   HydroRIVERS_v10_eu.shp (+ .shx .dbf .prj)
├── madenler.csv
├── obsidiyen.csv
└── osm/           gis_osm_roads_free_1.shp, gis_osm_places_free_1.shp (+ eşlikçi dosyaları)
```
Yükseklik verisi (DEM) için hiçbir şey indirmen gerekmez; kod internetten kendisi okur.
Çalıştırınca Google bir izin penceresi açar; kendi hesabınla onay ver.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
HAM = '/content/drive/MyDrive/tubitak-veri'
os.makedirs(HAM, exist_ok=True)
!rm -rf veri/ham && ln -s "$HAM" veri/ham
print("Drive'daki katmanlar:")
!find -L veri/ham -maxdepth 2 -name "*.shp" -o -maxdepth 2 -name "*.csv" -o -maxdepth 2 -name "*.gpkg"

## 3. Veri kontrolü
Yerleşim tablosunu denetler. **Hata varsa** GitHub'daki `veri/yerlesimler_temiz.csv` dosyasını düzelt ve 1. hücreden yeniden başla.

In [ ]:
!python 00_veri_kontrol.py

## 4. Değişkenleri hesapla
Her yerleşim için yükseklik, eğim, düz arazi oranı, görüş alanı ve mesafeleri hesaplar. Yerleşim sayısına göre 5–30 dakika sürebilir.

In [ ]:
!python 01_degisken_hesapla.py
import pandas as pd
pd.read_csv('sonuclar/degiskenler.csv').head(10)

**Dur ve kontrol et:** Tablodan 3 yerleşim seç. QGIS'te ya da Google Earth'te yüksekliklerine ve en yakın büyük akarsuya uzaklıklarına bak.
Değerler yakınsa devam et; çok farklıysa öğretmenine haber ver.

## 5. Ön-kayıt kontrolü
`02_model.py`'yi çalıştırmadan **önce** `ON_KAYIT.md` doldurulup GitHub'a commit'lenmiş olmalı. Aşağıdaki hücre bunu kontrol eder.

In [ ]:
!git log --format="%h  %ad  %s" --date=iso -- ON_KAYIT.md | head -3
!grep -c "\[ \]" ON_KAYIT.md && echo "UYARI: ON_KAYIT.md'de doldurulmamış alanlar var. Önce onu tamamla." || echo "ON_KAYIT.md dolu görünüyor."


## 6. Analiz
İstatistik testleri, Random Forest, mekânsal doğrulama ve hipotez kararları (10–40 dakika).

In [ ]:
!python 02_model.py

## 7. Sonuçları indir
`sonuclar.zip` bilgisayarına iner. Zip'i aç ve **`sonuclar` klasörünü** GitHub'da *Add file → Upload files* ile sürükle-bırak yükle. Streamlit uygulaması birkaç dakika içinde sonuçları göstermeye başlar.

In [ ]:
!zip -qr sonuclar.zip sonuclar
from google.colab import files
files.download('sonuclar.zip')